# 01 — Extraction & Initial Profiling

**Owner:** Tech Lead / ETL Owner
**Input:** `data/raw/online_retail_II.xlsx` (downloaded from UCI — do NOT edit)
**Output:** A raw pandas DataFrame loaded into memory + a profiling report committed back to this notebook
**Rubric link:** Data Quality & ETL (15 marks)

## Purpose
1. Load the raw dataset from disk
2. Confirm structure matches what the data dictionary claims (rows, columns, types)
3. Profile the raw data: shape, dtypes, missing-value map, cardinality, describe(), sample rows
4. Document anomalies found — feed these into the cleaning strategy in `02_cleaning.ipynb`

## Deliverables in this notebook
- A printed profile that proves the dataset meets the spec (≥5K rows, ≥8 columns, raw)
- A written "Observations" markdown cell at the end with the issues we will address in 02


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw/online_retail_II.xlsx")
assert RAW_PATH.exists(), f"Raw dataset not found at {RAW_PATH}. Download from https://archive.ics.uci.edu/dataset/502/online+retail+ii and place here."


In [ ]:
# Load both sheets and concatenate
xl = pd.ExcelFile(RAW_PATH)
print("Sheets:", xl.sheet_names)

frames = [xl.parse(sheet_name=s).assign(_source_sheet=s) for s in xl.sheet_names]
df_raw = pd.concat(frames, ignore_index=True)
print("Shape:", df_raw.shape)
df_raw.head()

## Structural profile

In [ ]:
# Column types
df_raw.dtypes

In [ ]:
# Missing values per column
missing = df_raw.isna().sum().to_frame("missing")
missing["pct"] = (missing["missing"] / len(df_raw) * 100).round(2)
missing.sort_values("pct", ascending=False)

In [ ]:
# Numeric describe
df_raw.describe()

In [ ]:
# Categorical cardinality
for col in ["Invoice", "StockCode", "Description", "Country"]:
    print(f"{col}: {df_raw[col].nunique()} unique values")

In [ ]:
# Peek at potential issues we already suspect from the dictionary
# 1. Negative quantities / prices (returns)
print("Rows with negative Quantity:", (df_raw['Quantity'] < 0).sum())
print("Rows with Quantity == 0   :", (df_raw['Quantity'] == 0).sum())
print("Rows with Price < 0       :", (df_raw['Price'] < 0).sum())
print("Rows with Price == 0      :", (df_raw['Price'] == 0).sum())

# 2. Invoice prefixed with 'C' (cancellations)
print("Rows with 'C' invoice prefix:", df_raw['Invoice'].astype(str).str.startswith('C').sum())

# 3. Missing Customer ID
print("Rows missing Customer ID:", df_raw['Customer ID'].isna().sum(),
      f"({df_raw['Customer ID'].isna().mean()*100:.1f}%)")

In [ ]:
# Country distribution (top 10) — UK skew check
df_raw['Country'].value_counts().head(10)

In [ ]:
# Non-product StockCode check — look at codes that aren't pure numeric
non_numeric_codes = df_raw[~df_raw['StockCode'].astype(str).str.match(r'^\d{5}[A-Z]?$')]
non_numeric_codes['StockCode'].value_counts().head(20)

## Observations for cleaning

*(Fill this in as you run the profiling cells above. Examples of what to note:)*

- **Missing `Customer ID`**: ~X% of rows — strategy in 02 = flag with `is_registered`, keep rows for product-level KPIs, drop for customer-level KPIs.
- **Negative quantities**: Y rows — represent returns; flag via `is_return`, handle explicitly in net-revenue formula.
- **Non-product stock codes**: `POST`, `M`, `D`, `BANK CHARGES` etc. — filter for product analysis but keep for gross revenue.
- **Country inconsistencies**: `EIRE` vs `Ireland`, `Unspecified` — standardize via mapping in 02.
- **Duplicate descriptions across stock codes**: note which ones are true duplicates vs legitimate similarity.
- **Time range**: X to Y — check for any implausible dates.

These observations drive the cleaning decisions in `02_cleaning.ipynb`.
